# AML Trainer

Treino offline do classificador (Random Forest, Spark MLlib).
Lê `LI-Small_Trans.csv` de HDFS, faz feature engineering, treina e persiste o `PipelineModel` em HDFS.
O `PipelineModel` é depois carregado pelos consumers (Kafka e file-based) e aplicado em streaming.

In [1]:
import time
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    StructType, StructField,
    TimestampType, IntegerType, StringType, DoubleType,
)
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

SPARK_MASTER = "spark://10.84.128.47:7077"
HDFS_BASE = "hdfs://10.84.129.52:9000/aulas/fabricio_moreira/project"
TRAIN_PATH = f"{HDFS_BASE}/dataset/LI-Small_Trans.csv"  #Onde o dataset de treinamento está guardado
MODEL_PATH = f"{HDFS_BASE}/model/rf_aml_pipeline"       #Onde o modelo treinado (Random Forest para AML*) está guardado

spark = SparkSession.builder \
    .master(SPARK_MASTER) \
    .appName("AML Model Trainer") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

Py4JJavaError: An error occurred while calling None.org.apache.spark.sql.classic.SparkSession.
: java.lang.IllegalStateException: Cannot call methods on a stopped SparkContext.
This stopped SparkContext was created at:

org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:59)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:77)
java.base/jdk.internal.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:45)
java.base/java.lang.reflect.Constructor.newInstanceWithCaller(Constructor.java:500)
java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:481)
py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:247)
py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
py4j.Gateway.invoke(Gateway.java:238)
py4j.commands.ConstructorCommand.invokeConstructor(Constru

In [ ]:
aml_schema = StructType([
    StructField("Timestamp", TimestampType(), True),
    StructField("From Bank", IntegerType(), True),
    StructField("From Account", StringType(), True),
    StructField("To Bank", IntegerType(), True),
    StructField("To Account", StringType(), True),
    StructField("Amount Received", DoubleType(), True),
    StructField("Receiving Currency", StringType(), True),
    StructField("Amount Paid", DoubleType(), True),
    StructField("Payment Currency", StringType(), True),
    StructField("Payment Format", StringType(), True),
    StructField("Is Laundering", IntegerType(), True),
])

In [ ]:
df = spark.read \
    .option("header", True) \
    .option("timestampFormat", "yyyy/MM/dd HH:mm") \
    .schema(aml_schema) \
    .csv(TRAIN_PATH)

total = df.count()
positives = df.filter("`Is Laundering` = 1").count()
print(f"Linhas: {total:,}  Laundering: {positives:,} ({positives/total:.4%})")

In [ ]:
df.show(5)

## Feature engineering

Idêntica nos consumers — para evitar duplicação, está empacotada numa função que será reproduzida nos notebooks dos consumers.

In [ ]:
def add_features(d):
    return (
        d.withColumn("log_amt_paid", F.log1p(F.col("Amount Paid")))
         .withColumn("log_amt_recv", F.log1p(F.col("Amount Received")))
         .withColumn("amt_diff", F.abs(F.col("Amount Paid") - F.col("Amount Received")))
         .withColumn("same_bank", (F.col("From Bank") == F.col("To Bank")).cast("int"))
         .withColumn("ccy_mismatch", (F.col("Receiving Currency") != F.col("Payment Currency")).cast("int"))
         .withColumn("hour", F.hour("Timestamp"))
    )

featured = add_features(df) \
    .withColumnRenamed("Is Laundering", "label") \
    .na.fill({"hour": 0})

train, test = featured.randomSplit([0.8, 0.2], seed=42)
train.cache()
print(f"Train: {train.count():,}  Test: {test.count():,}")

In [ ]:
# Pesos de classe para o forte desbalanceamento (laundering << 1%)
pos = train.filter("label = 1").count()
neg = train.filter("label = 0").count()
w_pos = neg / (pos + neg)
w_neg = pos / (pos + neg)
print(f"weight pos={w_pos:.4f}  neg={w_neg:.4f}")

train_w = train.withColumn(
    "class_weight",
    F.when(F.col("label") == 1, F.lit(w_pos)).otherwise(F.lit(w_neg)),
)

In [ ]:
train.show(2)

In [ ]:
train_w.show(2)

In [ ]:
cat_cols = ["Receiving Currency", "Payment Currency", "Payment Format"]
num_cols = ["log_amt_paid", "log_amt_recv", "amt_diff", "same_bank", "ccy_mismatch", "hour", "From Bank", "To Bank"]

indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in cat_cols]
encoders = [OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe", handleInvalid="keep") for c in cat_cols]

assembler = VectorAssembler(
    inputCols=num_cols + [f"{c}_ohe" for c in cat_cols],
    outputCol="features",
    handleInvalid="keep",
)

rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    weightCol="class_weight",
    numTrees=50,
    maxDepth=10,
    seed=42,
)

pipeline = Pipeline(stages=indexers + encoders + [assembler, rf])

In [ ]:
t0 = time.time()
model = pipeline.fit(train_w)
train_time = time.time() - t0
print(f"Tempo de treino: {train_time:.1f}s")

In [ ]:
preds = model.transform(test)

auc = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC",
).evaluate(preds)
f1 = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1",
).evaluate(preds)

print(f"AUC: {auc:.4f}   F1: {f1:.4f}\n")
print("Matriz de confusão:")
preds.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

In [ ]:
model.write().overwrite().save(MODEL_PATH)
print(f"Modelo guardado em {MODEL_PATH}")